# D-RECIPE: Dynamic Temporal Knowledge Graph Forecasting

**Continual LLM-based TKG completion with streaming edge ingestion.**

> Before running: go to **Runtime → Change runtime type** and select **GPU (T4 or A100)**.

---
**Steps in this notebook:**
1. Mount Google Drive (for saving results)
2. Clone repo & install dependencies
3. Authenticate HuggingFace (for LLaMA access)
4. Download & prepare dataset
5. Run training
6. Run inference & evaluation
7. Generate plots

## 1. Mount Google Drive
Saves checkpoints and results to your Drive so they survive session resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/D-RECIPE'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Results will be saved to: {DRIVE_DIR}')

## 2. Clone Repository & Install Dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/saniemacdube93/forecast.git'  # update if different
REPO_DIR = '/content/forecast'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest changes')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!ls

In [ ]:
# Install dependencies (~3-5 minutes on first run)
!pip install -q -r requirements_dynamic.txt
print('Dependencies installed.')

In [ ]:
# Verify GPU is available
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 3. HuggingFace Authentication

LLaMA-2 and LLaMA-3 require accepting the licence on HuggingFace before downloading.

- LLaMA-2: https://huggingface.co/meta-llama/Llama-2-7b-hf  
- LLaMA-3: https://huggingface.co/meta-llama/Meta-Llama-3-8B

Paste your HuggingFace **read token** below (Settings → Access Tokens).

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Option A: load from Colab Secrets (recommended — add HF_TOKEN in the key icon sidebar)
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in via Colab secret.')
except Exception:
    # Option B: paste token directly
    hf_token = ''  # <-- paste your token here
    if hf_token:
        login(token=hf_token, add_to_git_credential=False)
        print('Logged in via pasted token.')
    else:
        print('WARNING: No HuggingFace token found. LLaMA download will fail.')

## 4. Download & Prepare Data

Data is hosted on Google Drive. Download and extract it, then run preprocessing.

In [ ]:
# Download data from Google Drive (ID from README link)
# Source: https://drive.google.com/drive/folders/1kdo_pn6PDig7SJ61feugRDwckcTZ8KHX

!pip install -q gdown

DATA_DIR = '/content/forecast/data/original'
os.makedirs(DATA_DIR, exist_ok=True)

# Download each dataset folder
DATASETS = {
    'icews14': '1kdo_pn6PDig7SJ61feugRDwckcTZ8KHX',  # update folder IDs if needed
}

# If you have the Drive folder shared, use gdown to download the whole folder:
!gdown --folder https://drive.google.com/drive/folders/1kdo_pn6PDig7SJ61feugRDwckcTZ8KHX -O {DATA_DIR} --remaining-ok
!ls {DATA_DIR}

In [ ]:
# Preprocess data (retrieval step — same as original RECIPE-TKG)
DATASET = 'icews14'  # change to: icews18, GDELT, YAGO

%cd /content/forecast/data_utils
!python retrieve.py --dataset {DATASET} --retrieve_type weighted
%cd /content/forecast

## 5. Configuration

Set your experiment parameters here. These are passed as CLI flags to `dynamic_main.py`.

In [ ]:
# ── Experiment settings ──────────────────────────────────────────────────────
DATASET        = 'icews14'                          # icews14 | icews18 | GDELT | YAGO
MODEL_NAME     = 'meta-llama/Llama-2-7b-hf'        # or meta-llama/Meta-Llama-3-8B
DATA_PATH      = './data/original/'
RESULTS_DIR    = f'{DRIVE_DIR}/results/{DATASET}_llama2'

# ── Stream simulation ────────────────────────────────────────────────────────
STREAM_CHUNK_SIZE = 500
SEED_CHUNKS       = 5
N_CHUNKS          = 20
EPOCHS            = 5

# ── Continual learning weights ───────────────────────────────────────────────
EWC_LAMBDA   = 0.1
KD_GAMMA     = 0.3
KD_TEMP      = 2.0
REPLAY_RATIO = 0.3
BUFFER_SIZE  = 5000

# ── Hardware ──────────────────────────────────────────────────────────────────
DEVICE             = 'cuda'   # auto | cuda | cpu
MICRO_BATCH_SIZE   = 2
GRAD_CHECKPOINTING = 1

os.makedirs(RESULTS_DIR, exist_ok=True)
print('Config ready. Results dir:', RESULTS_DIR)

## 6. Train D-RECIPE

In [ ]:
# Quick debug run first — verifies the pipeline end-to-end in ~5 minutes
!python dynamic_main.py \
    --DATASET {DATASET} \
    --MODEL_NAME {MODEL_NAME} \
    --DATA_PATH {DATA_PATH} \
    --RESULTS_DIR {RESULTS_DIR}/debug \
    --STREAM_CHUNK_SIZE 100 \
    --SEED_CHUNKS 1 \
    --N_CHUNKS 3 \
    --EPOCHS 1 \
    --DEVICE {DEVICE} \
    --DEBUG

In [ ]:
# Full training run
!python dynamic_main.py \
    --DATASET {DATASET} \
    --MODEL_NAME {MODEL_NAME} \
    --DATA_PATH {DATA_PATH} \
    --RESULTS_DIR {RESULTS_DIR} \
    --STREAM_CHUNK_SIZE {STREAM_CHUNK_SIZE} \
    --SEED_CHUNKS {SEED_CHUNKS} \
    --N_CHUNKS {N_CHUNKS} \
    --EPOCHS {EPOCHS} \
    --EWC_LAMBDA {EWC_LAMBDA} \
    --KD_GAMMA {KD_GAMMA} \
    --KD_TEMPERATURE {KD_TEMP} \
    --REPLAY_RATIO {REPLAY_RATIO} \
    --BUFFER_SIZE {BUFFER_SIZE} \
    --DEVICE {DEVICE} \
    --MICRO_BATCH_SIZE {MICRO_BATCH_SIZE} \
    --GRADIENT_CHECKPOINTING {GRAD_CHECKPOINTING}

## 7. Inference & Evaluation

In [ ]:
CHECKPOINT = f'{RESULTS_DIR}/checkpoints/final_model.pt'
OUTPUT_FILE = f'{RESULTS_DIR}/predictions.json'

!python dynamic_inference.py \
    --DATASET {DATASET} \
    --CHECKPOINT {CHECKPOINT} \
    --MODEL_NAME {MODEL_NAME} \
    --DATA_PATH {DATA_PATH} \
    --OUTPUT_FILE {OUTPUT_FILE} \
    --DEVICE {DEVICE} \
    --EVAL

## 8. Generate Plots

In [ ]:
PLOTS_DIR = f'{RESULTS_DIR}/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

METRICS_FILE = f'{RESULTS_DIR}/metrics.json'

!python visualize_results.py \
    --results_file {METRICS_FILE} \
    --output_dir {PLOTS_DIR}

In [ ]:
# Display all generated plots inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

plot_files = sorted(glob.glob(f'{PLOTS_DIR}/*.png'))
if not plot_files:
    # Fall back to the pre-generated plots in the repo
    plot_files = sorted(glob.glob('/content/forecast/results/plots/*.png'))

for path in plot_files:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(mpimg.imread(path))
    ax.axis('off')
    ax.set_title(os.path.basename(path), fontsize=11)
    plt.tight_layout()
    plt.show()

## 9. Ablation Study (Optional)

In [ ]:
# Run a single ablation — remove one component at a time
# Options: no_ewc | no_replay | no_kd | no_filtering | naive_finetune

ABLATION = 'no_ewc'

!python dynamic_main.py \
    --DATASET {DATASET} \
    --MODEL_NAME {MODEL_NAME} \
    --DATA_PATH {DATA_PATH} \
    --RESULTS_DIR {DRIVE_DIR}/results/ablation_{ABLATION} \
    --STREAM_CHUNK_SIZE {STREAM_CHUNK_SIZE} \
    --SEED_CHUNKS {SEED_CHUNKS} \
    --N_CHUNKS {N_CHUNKS} \
    --EPOCHS {EPOCHS} \
    --ABLATION {ABLATION} \
    --DEVICE {DEVICE}

## 10. Generate Mock Plots (No Training Required)

Use this to preview all 11 figure types using the paper's baseline numbers — no GPU or model download needed.

In [ ]:
MOCK_PLOTS_DIR = f'{DRIVE_DIR}/mock_plots'
os.makedirs(MOCK_PLOTS_DIR, exist_ok=True)

!python visualize_results.py --mock --output_dir {MOCK_PLOTS_DIR}

# Display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, os

for path in sorted(glob.glob(f'{MOCK_PLOTS_DIR}/*.png')):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(mpimg.imread(path))
    ax.axis('off')
    ax.set_title(os.path.basename(path), fontsize=11)
    plt.tight_layout()
    plt.show()